# 下一季度指引与预期来源核查

沿105次已核对财报日历读取之后首个真实交易日晨报。只抽取公司报道段落，关键词命中不等于季度/数值对齐，不把“高于预期”补成精确数值。当前阶段0拟合、0回测。

In [1]:
from pathlib import Path
import os,json,hashlib
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import qlib,pandas as pd,yaml
from qlib.data import D
from qlib.workflow import R
from src.data.guidance_newsletters import collect_guidance_newsletters
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/guidance_newsletter_source_20261013';plan=json.loads((card/'plan.json').read_text())
assert hashlib.sha256((root/'research/issuer_actuals_qualification_20261013/plan.json').read_bytes()).hexdigest()==plan['issuer_calendar_plan_sha256']
qlib.init(provider_uri=str(root/plan['price_provider']),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(freq='day'))
for e in plan['events']:assert calendar[calendar>pd.Timestamp(e['report_date'])][0]==pd.Timestamp(e['newsletter_date'])
raw=root/plan['raw_cache'];manifest=collect_guidance_newsletters(raw,plan)
frame=pd.DataFrame(manifest['entries']);frame['company_block_count']=frame.paragraphs.map(len)
frame['needs_semantic_review']=frame.guidance_word_present.fillna(False)&frame.quarter_word_present.fillna(False)&frame.estimate_word_present.fillna(False)
modified=pd.to_datetime(frame.modified_at,utc=True,errors='coerce');created=pd.to_datetime(frame.created_at,utc=True,errors='coerce')
latest=pd.concat([modified,created],axis=1).max(axis=1)
frame['document_clock_before_decision']=frame.clock_timezone_known & latest.notna() & latest.dt.tz_convert('America/New_York').dt.tz_localize(None).le(pd.to_datetime(frame.newsletter_date)+pd.Timedelta(hours=9,minutes=46))
audit={'expected_documents':len(plan['events']),'downloaded_and_read_documents':len(frame),'failures':manifest['failures'],'company_block_present':int(frame.company_block_count.gt(0).sum()),'guidance_quarter_estimate_keyword_candidates':int(frame.needs_semantic_review.sum()),'document_clock_before_decision':int(frame.document_clock_before_decision.sum()),'numeric_consensus_pairs_qualified':0,'target_quarter_pairs_qualified':0,'historical_publication_certified':False,'model_fits':0,'backtests':0}
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],recorder_name='post_release_guidance_source_inventory',uri=str(tracking)):
    rec=R.get_recorder();rec.log_params(plan_sha256=hashlib.sha256((card/'plan.json').read_bytes()).hexdigest(),model_fits=0,backtests=0)
    rec.save_objects(artifact_path='sources',manifest=manifest,pdfs={p.name:p.read_bytes() for p in raw.glob('*.pdf')},parser=(root/'src/data/guidance_newsletters.py').read_text())
    rec.save_objects(artifact_path='analysis',paragraphs=frame,audit=audit,calendar=calendar)
    rec.log_metrics(registered_artifact_count=6,source_documents=len(frame))
summary={'experiment':plan['experiment'],'decision':'needs_target_quarter_and_consensus_pair_review','source_audit':audit,'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(tracking),'recorder_local_dir':str(tracking/str(rec.experiment_id)/rec.id),'provider_uri':str(root/plan['price_provider']),'as_of':plan['as_of'],'region':'us','qlib_docs':'https://qlib.readthedocs.io/en/latest/component/workflow.html','objective_achieved':False,'goal_remains_active':True,'promotion_eligible':False,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['sources/*','analysis/*']}}
artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),'bytes':p.stat().st_size} for p in sorted(artifacts.rglob('*')) if p.is_file()]},indent=2)+'\n')
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id))
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':plan['experiment'],'provider_uri':summary['provider_uri'],'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']}));frame.drop(columns=['paragraphs']).to_csv(card/'coverage.csv',index=False)
audit


{'expected_documents': 105,
 'downloaded_and_read_documents': 104,
 'failures': [{'symbol': 'INTC',
   'report_date': '2019-10-24',
   'newsletter_date': '2019-10-25',
   'target_fiscal_quarter': 'FY2019Q4',
   'url': 'https://share.refinitiv.com/assets/newsletters/Morning_News_Call/MNCGeneric_US_10252019.pdf',
   'path': '10252019.pdf',
   'error': 'curl: (22) The requested URL returned error: 404\n'}],
 'company_block_present': 100,
 'guidance_quarter_estimate_keyword_candidates': 95,
 'document_clock_before_decision': 104,
 'numeric_consensus_pairs_qualified': 0,
 'target_quarter_pairs_qualified': 0,
 'historical_publication_certified': False,
 'model_fits': 0,
 'backtests': 0}

# 财报后晨报中的下一季度指引来源

沿105次固定财报日历，取得104份随后首个Qlib真实交易日的晨报；Intel2019-10-24后的2019-10-25文件返回404并保留。100份识别到对应公司报道段落，95份同时包含指引、季度、预期相关关键词。关键词只是待核查线索，数值配对与目标季度认证均为0，不是95个有效信号。

早期股票代码括号和句号分隔符导致部分段落未识别，现已修复；同时排除了独立分析师目标价推荐段落，避免把股价目标混成营收预期。修复前的69个段落版本及NAS证据保存在qualification_attempts.json。全部报道正文与PDF保存在Recorder，根目录仅保留覆盖表和引用。

104份文件均有明确时区，文档创建/修改时刻早于对应交易日09:46，创建日期与拟用晨报日期一致。这仍是文档时间的一致性检查，不证明历史首次公开或接收；原站证书过期的公开文件读取限制继续记录。没有从搜索摘要或“高于预期百分比”的描述反推缺失的精确分母。

抽查说明，同一篇公司报道可能同时含已公布季度的实际营收、下一季度指引、全年展望、分部收入预期以及EPS。后续必须逐项确认目标季度、指标、单位及所引用的预期，不能仅凭出现“forecast”和“estimates”计算差值；只有方向描述时单独记录，不当作精确金额。

本轮0模型、0回测，没有新增收益，生产SIM未改。决定needs_target_quarter_and_consensus_pair_review。下一步对这100个段落完成语义与公司原始指引核对，保留缺失；达到数据合同后再冻结有限信息实验。稳定净SOXX超额目标仍未达成，保持active。
